# Tiny Transformer adaptation

Run a baseline trace, inspect Attention evidence and pack the three projections into one linear operator. Compare identical FP32 inputs on the local CPU.

In [ ]:
from pathlib import Path
import sys
project = Path.cwd()
if not (project / "htp_insight").is_dir():
    project = project.parent
sys.path.insert(0, str(project))
import torch
from torch import nn
from copy import deepcopy
from htp_insight.core import Core
from examples.demo import baseline, candidate

core = Core()
original, x, initial = baseline(core)
[(row["module"], row["total_us"]) for row in core.get_hotspots(initial["id"])["hotspots"]]

In [ ]:
evidence = core.explain_hotspot(initial["id"], "attention")
print(evidence["total_us"], evidence["share"], evidence["example_evidence"]["id"])
print(evidence["hypothesis"])

In [ ]:
adapted = deepcopy(original)
attention = adapted.attention
q, k, v = attention.q, attention.k, attention.v
attention.qkv = nn.Linear(q.in_features, 3 * q.out_features)
with torch.no_grad():
    attention.qkv.weight.copy_(torch.cat([q.weight, k.weight, v.weight]))
    attention.qkv.bias.copy_(torch.cat([q.bias, k.bias, v.bias]))
del attention.q, attention.k, attention.v
adapted.eval()
with torch.inference_mode():
    torch.testing.assert_close(original(x), adapted(x), atol=1e-5, rtol=1e-4)

In [ ]:
result, torch_check = candidate(core, original, x, initial, adapted)
assert torch_check["passed"] and result["numeric"]["passed"]
display(result["latency"], result["numeric"], result["graph_diff"], result["deep_link"])